# Graph Neural Networks: Complete Learning Guide
## Module 1: Fundamentals & Theory

### Course Overview
This comprehensive notebook covers Graph Neural Networks (GNNs) from first principles through practical implementation. Each section builds mathematical foundations, implements concepts from scratch, and demonstrates real-world applications.

**Learning Objectives:**
- Understand graph theory fundamentals
- Master message passing frameworks
- Implement GCN, GAT, and GraphSAGE architectures
- Apply GNNs to node classification and link prediction tasks
- Visualize and interpret learned representations

**Prerequisites:**
- Linear algebra fundamentals
- Basic neural network knowledge
- Python programming proficiency

---

## 📚 Table of Contents
1. Graph Theory Fundamentals
2. Graph Representations & Data Structures
3. Message Passing Framework
4. Graph Convolutional Networks (GCN)
5. Spectral vs. Spatial Methods
6. GraphSAGE Architecture
7. Graph Attention Networks (GAT)
8. Node Classification Pipeline
9. Link Prediction Tasks
10. Visualization & Interpretation

# Section 1: Graph Theory Fundamentals

## 1.1 Core Concepts

### What is a Graph?

A **graph** $G = (V, E)$ is a fundamental data structure consisting of:
- **Vertices (Nodes)** $V = \{v_1, v_2, ..., v_n\}$: Entities or objects in the system
- **Edges** $E \subseteq V \times V$: Relationships or connections between vertices

### Graph Properties

#### Degree of a Node

The **degree** of node $v_i$ is the number of edges connected to it:

$$d_i = |\\{(v_i, v_j) : (v_i, v_j) \in E\\}|$$

For **directed graphs**, we distinguish:
- **In-degree**: Number of incoming edges
- **Out-degree**: Number of outgoing edges

#### Adjacency Matrix

The **adjacency matrix** $A$ of a graph is an $n \times n$ matrix where:

$$A_{ij} = \begin{cases} 
1 & \text{if } (v_i, v_j) \in E \\
0 & \text{otherwise}
\end{cases}$$

For **undirected graphs**: $A = A^T$ (symmetric)  
For **weighted graphs**: $A_{ij}$ represents edge weight $w_{ij}$

#### Degree Matrix

The **degree matrix** $D$ is a diagonal matrix:

$$D = \begin{pmatrix}
d_1 & 0 & \cdots & 0 \\
0 & d_2 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & d_n
\end{pmatrix}$$

#### Laplacian Matrix

The **Graph Laplacian** is a crucial matrix in spectral graph theory:

$$L = D - A$$

**Normalized Laplacian**:
$$L_{norm} = I - D^{-1/2}AD^{-1/2}$$

This matrix encodes structural information about the graph topology.

### Spectral Properties

The eigenvalues and eigenvectors of $L$ reveal important graph characteristics:

$$L v_i = \lambda_i v_i$$

- **Multiplicity of 0 eigenvalue** = Number of connected components
- **Second-smallest eigenvalue (Fiedler value)** $\lambda_2$ = Algebraic connectivity
- **Spectral gap** $= \lambda_2$ indicates how well-connected the graph is

---

## 1.2 Graph Types and Examples

In [ ]:
# Import essential libraries for GNN development
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from typing import Tuple, List, Dict, Union
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

# Configure visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully!")
print("Ready to explore Graph Neural Networks!\n")

In [ ]:
# ============================================================================
# EXAMPLE 1: Simple Graph Construction and Analysis
# ============================================================================

# Create a small undirected graph
G_simple = nx.Graph()

# Add nodes (vertices)
G_simple.add_nodes_from([0, 1, 2, 3, 4])

# Add edges (connections)
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4)]
G_simple.add_edges_from(edges)

print("=" * 70)
print("EXAMPLE 1: Simple Graph Analysis")
print("=" * 70)
print(f"\n📊 Graph Properties:")
print(f"  • Number of nodes: {G_simple.number_of_nodes()}")
print(f"  • Number of edges: {G_simple.number_of_edges()}")
print(f"  • Nodes: {list(G_simple.nodes())}")
print(f"  • Edges: {list(G_simple.edges())}")

# Compute degree of each node
print(f"\n📈 Node Degrees:")
degrees = dict(G_simple.degree())
for node, degree in degrees.items():
    print(f"  • Node {node}: degree = {degree}")

# Compute average degree
avg_degree = sum(degrees.values()) / len(degrees)
print(f"  • Average degree: {avg_degree:.2f}")

# Get adjacency matrix
A_simple = nx.to_numpy_array(G_simple, dtype=int)
print(f"\n🔢 Adjacency Matrix A:")
print("   (rows and columns represent nodes 0-4)")
print(A_simple.astype(int))

# Compute degree matrix
D_simple = np.diag(np.array(G_simple.degree())[:, 1])
print(f"\n🔢 Degree Matrix D (diagonal):")
print(D_simple.astype(int))

# Compute Laplacian matrix
L_simple = D_simple - A_simple
print(f"\n🔢 Laplacian Matrix L = D - A:")
print(L_simple.astype(int))

# Compute eigenvalues of Laplacian
eigenvalues, eigenvectors = np.linalg.eig(L_simple)
eigenvalues = np.sort(eigenvalues.real)
print(f"\n📊 Laplacian Eigenvalues (sorted):")
for i, eig_val in enumerate(eigenvalues):
    print(f"  • λ_{i} = {eig_val:.6f}")

print(f"\n  ➜ Number of connected components: {int(np.round(eigenvalues[0]))} (multiplicity of λ₀=0)")

In [ ]:
# Visualize the simple graph
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Graph visualization
ax1 = axes[0]
pos = nx.spring_layout(G_simple, seed=42, k=2, iterations=50)
nx.draw_networkx_nodes(G_simple, pos, node_color='lightblue', node_size=1000, ax=ax1)
nx.draw_networkx_edges(G_simple, pos, width=2, ax=ax1)
nx.draw_networkx_labels(G_simple, pos, font_size=12, font_weight='bold', ax=ax1)
ax1.set_title("Graph Structure Visualization", fontsize=14, fontweight='bold')
ax1.axis('off')

# Plot 2: Adjacency matrix heatmap
ax2 = axes[1]
im = ax2.imshow(A_simple, cmap='Blues', aspect='auto')
ax2.set_xticks(range(5))
ax2.set_yticks(range(5))
ax2.set_xticklabels(range(5))
ax2.set_yticklabels(range(5))
ax2.set_xlabel("Node j", fontsize=12)
ax2.set_ylabel("Node i", fontsize=12)
ax2.set_title("Adjacency Matrix A (Binary)", fontsize=14, fontweight='bold')

# Add text annotations
for i in range(5):
    for j in range(5):
        text = ax2.text(j, i, int(A_simple[i, j]), ha="center", va="center", 
                       color="white" if A_simple[i, j] > 0.5 else "black", fontweight='bold')

plt.colorbar(im, ax=ax2, label='Connection')
plt.tight_layout()
plt.show()

print("\n✓ Graph visualization complete!")

# Section 2: Graph Representations & Data Structures

## 2.1 Graph Representation Methods

In practice, we represent graphs using different data structures optimized for different operations:

### 1. **Adjacency Matrix** (Dense representation)
- **Pros**: Easy to understand, matrix operations
- **Cons**: O(n²) memory, inefficient for sparse graphs
- **Use case**: Small graphs, dense connections

### 2. **Adjacency List** (Sparse representation)
- **Pros**: O(V+E) memory, efficient for sparse graphs
- **Cons**: Less efficient for dense graphs
- **Use case**: Most real-world graphs (social networks, web graphs)

### 3. **Edge List** (Sparse representation)
- **Pros**: Compact for large sparse graphs
- **Cons**: Slower neighborhood lookup
- **Use case**: Graph algorithms, distributed processing

## 2.2 Computational Complexity Comparison

| Operation | Adjacency Matrix | Adjacency List | Edge List |
|-----------|-----------------|----------------|-----------|
| Check edge (i,j) | O(1) | O(degree(i)) | O(E) |
| Add node | O(V²) | O(1) | O(1) |
| Add edge | O(1) | O(1) | O(1) |
| Neighbors of i | O(V) | O(degree(i)) | O(E) |
| Space | O(V²) | O(V+E) | O(E) |

In [ ]:
# ============================================================================
# IMPLEMENTATION: Graph Data Structures
# ============================================================================

class Graph:
    """
    A flexible graph class supporting multiple representations.
    
    This class demonstrates how to work with graphs efficiently using
    different internal representations optimized for various operations.
    """
    
    def __init__(self, num_nodes: int, directed: bool = False):
        """
        Initialize a graph.
        
        Args:
            num_nodes: Number of nodes in the graph
            directed: Whether the graph is directed (default: undirected)
        """
        self.num_nodes = num_nodes
        self.directed = directed
        
        # Adjacency list representation (most efficient for sparse graphs)
        # Dict mapping: node_id -> list of neighbor nodes
        self.adj_list: Dict[int, List[int]] = {i: [] for i in range(num_nodes)}
        
        # Edge storage (for iteration)
        self.edges: List[Tuple[int, int]] = []
        
        # Node features (can store attributes like embeddings)
        self.node_features: Union[np.ndarray, None] = None
        
        # Edge weights (optional)
        self.edge_weights: Dict[Tuple[int, int], float] = {}
    
    def add_edge(self, u: int, v: int, weight: float = 1.0):
        """
        Add an edge between two nodes.
        
        Args:
            u: Source node
            v: Target node
            weight: Edge weight (default: 1.0)
        """
        # Add to adjacency list
        self.adj_list[u].append(v)
        self.edges.append((u, v))
        self.edge_weights[(u, v)] = weight
        
        # For undirected graphs, add reverse edge
        if not self.directed:
            self.adj_list[v].append(u)
            self.edge_weights[(v, u)] = weight
    
    def get_neighbors(self, node: int) -> List[int]:
        """
        Get all neighbors of a node.
        
        Time Complexity: O(1) - direct dictionary lookup
        
        Args:
            node: The node to query
            
        Returns:
            List of neighbor node indices
        """
        return self.adj_list[node]
    
    def degree(self, node: int) -> int:
        """
        Get the degree (number of neighbors) of a node.
        
        Time Complexity: O(1)
        
        Args:
            node: The node to query
            
        Returns:
            Degree of the node
        """
        return len(self.adj_list[node])
    
    def to_adjacency_matrix(self) -> np.ndarray:
        """
        Convert to dense adjacency matrix.
        
        Returns:
            n × n adjacency matrix
        """
        A = np.zeros((self.num_nodes, self.num_nodes))
        for u, v in self.edges:
            weight = self.edge_weights.get((u, v), 1.0)
            A[u, v] = weight
        
        return A
    
    def to_edge_list(self) -> np.ndarray:
        """
        Convert to edge list format.
        
        Returns:
            2 × E array where columns are (source, target) pairs
        """
        if len(self.edges) == 0:
            return np.array([]).reshape(2, 0)
        
        edge_list = np.array(self.edges).T
        return edge_list
    
    def compute_laplacian(self) -> np.ndarray:
        """
        Compute the graph Laplacian matrix L = D - A.
        
        Mathematical derivation:
        - D: Degree matrix (diagonal) where D[i,i] = degree(i)
        - A: Adjacency matrix
        - L = D - A encodes graph structure
        
        Returns:
            Laplacian matrix
        """
        A = self.to_adjacency_matrix()
        
        # Compute degree of each node
        degrees = np.array([self.degree(i) for i in range(self.num_nodes)])
        D = np.diag(degrees)
        
        # Laplacian = Degree - Adjacency
        L = D - A
        
        return L
    
    def compute_normalized_laplacian(self) -> np.ndarray:
        """
        Compute normalized Laplacian L_norm = I - D^(-1/2) A D^(-1/2).
        
        This normalization is crucial for spectral methods because:
        1. It scales eigenvalues to [0, 2]
        2. Makes the matrix more numerically stable
        3. Helps with convergence in iterative algorithms
        
        Returns:
            Normalized Laplacian matrix
        """
        A = self.to_adjacency_matrix()
        
        # Compute degree
        degrees = np.array([self.degree(i) for i in range(self.num_nodes)])
        
        # Avoid division by zero (isolated nodes)
        degrees_inv_sqrt = np.zeros_like(degrees, dtype=float)
        mask = degrees > 0
        degrees_inv_sqrt[mask] = 1.0 / np.sqrt(degrees[mask])
        
        D_inv_sqrt = np.diag(degrees_inv_sqrt)
        
        # Normalized Laplacian
        I = np.eye(self.num_nodes)
        L_norm = I - D_inv_sqrt @ A @ D_inv_sqrt
        
        return L_norm

# ============================================================================
# Example: Create a graph from EXAMPLE 1 and demonstrate representations
# ============================================================================

print("\n" + "=" * 70)
print("IMPLEMENTATION: Graph Representations")
print("=" * 70)

# Create graph with same structure as EXAMPLE 1
G = Graph(num_nodes=5, directed=False)
for u, v in edges:
    G.add_edge(u, v, weight=1.0)

print("\n1️⃣ ADJACENCY LIST REPRESENTATION:")
print("   (Most efficient for sparse graphs)")
for node in range(5):
    print(f"   Node {node}: neighbors = {G.get_neighbors(node)}, degree = {G.degree(node)}")

print("\n2️⃣ EDGE LIST REPRESENTATION:")
edge_list = G.to_edge_list()
print(f"   Shape: {edge_list.shape}")
print(f"   Edges:\n{edge_list}")

print("\n3️⃣ ADJACENCY MATRIX REPRESENTATION:")
A_dense = G.to_adjacency_matrix()
print(f"   Shape: {A_dense.shape}")
print(f"   Matrix:\n{A_dense.astype(int)}")

print("\n4️⃣ LAPLACIAN MATRIX:")
L = G.compute_laplacian()
print(f"   L = D - A:\n{L.astype(int)}")

print("\n5️⃣ NORMALIZED LAPLACIAN:")
L_norm = G.compute_normalized_laplacian()
print(f"   L_norm = I - D^(-1/2) A D^(-1/2):")
print(L_norm)

# Section 3: Message Passing Framework

## 3.1 The General Message Passing Neural Network

The message passing framework is the **unified abstraction** for all GNNs. It works in two phases:

### Mathematical Formulation

For each node $v_i$ and each layer $\ell$:

**Message Computation (Aggregation Phase):**

$$m_i^{(\ell)} = \text{AGGREGATE}^{(\ell)} \\left( \\{ h_j^{(\ell-1)} : j \in \mathcal{N}(i) \\} \\right)$$

Where:
- $h_j^{(\ell-1)}$ is the hidden state of neighbor $j$ from previous layer
- $\mathcal{N}(i)$ is the set of neighbors of node $i$
- AGGREGATE is a permutation-invariant function (e.g., sum, mean, max)

**Update Phase:**

$$h_i^{(\ell)} = \text{UPDATE}^{(\ell)} \\left( h_i^{(\ell-1)}, m_i^{(\ell)} \\right)$$

Where UPDATE is typically a learnable neural network function (e.g., MLP)

### Key Insight: Permutation Invariance

The AGGREGATE function must be permutation-invariant because:
- The order of neighbors is arbitrary
- We want consistent results regardless of neighbor ordering
- Examples: SUM, MEAN, MAX, CONCATENATE

### Information Flow Architecture

In [ ]:
# Visualize message passing with mermaid-style ASCII diagram
print("\n" + "=" * 70)
print("MESSAGE PASSING VISUALIZATION")
print("=" * 70)

message_passing_diagram = """
LAYER ℓ-1: Node Features h^(ℓ-1)
═════════════════════════════════════

    [h₀]          [h₁]          [h₂]
     │  \\        /  │  \\      /  │
     │   \\      /   │   \\    /   │
     │    \\    /    │    \\  /    │
     
                    ↓
                
PHASE 1: MESSAGE AGGREGATION
═════════════════════════════════════

For each node i:
    - Collect messages from all neighbors: {h_j : j ∈ N(i)}
    - AGGREGATE them into m_i using symmetric function
    
Example for node 1:
    m₁ = AGGREGATE({h₀, h₂, h₃})
       = MEAN([h₀, h₂, h₃])     [if using mean aggregation]
       = or SUM, MAX, etc.
    
                    ↓

PHASE 2: STATE UPDATE
═════════════════════════════════════

For each node i:
    h_i^(ℓ) = UPDATE(h_i^(ℓ-1), m_i)
             = σ(W · [h_i^(ℓ-1) || m_i])
    
Where:
    - UPDATE is a learnable neural function (MLP)
    - σ is activation (ReLU, etc.)
    - W is weight matrix
    - [·||·] denotes concatenation

                    ↓

LAYER ℓ: Updated Node Features h^(ℓ)
═════════════════════════════════════

    [h₀']          [h₁']          [h₂']
     
Each node's new feature incorporates:
✓ Its own previous features
✓ Information from 1-hop neighbors
✓ Recursively from k-hop neighbors after k layers
"""

print(message_passing_diagram)

In [ ]:
# ============================================================================
# IMPLEMENTATION: Message Passing Neural Network Framework
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class MessagePassingLayer(nn.Module):
    """
    A generic message passing layer implementing the core GNN update rule.
    
    This is the foundation layer used in most GNN architectures.
    
    Mathematical operations:
    1. Message computation: m_i = AGGREGATE({h_j : j ∈ N(i)})
    2. Node update: h_i' = UPDATE(h_i, m_i)
    """
    
    def __init__(self, in_channels: int, out_channels: int, aggregation: str = 'mean'):
        """
        Initialize message passing layer.
        
        Args:
            in_channels: Dimension of input node features
            out_channels: Dimension of output node features
            aggregation: Type of aggregation function ('mean', 'sum', 'max')
        """
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.aggregation = aggregation
        
        # Linear transformation for message preparation
        # This learns how to transform neighbor features before aggregation
        self.message_func = nn.Linear(in_channels, out_channels)
        
        # Linear transformation for node update
        # This learns how to combine own features with aggregated messages
        self.update_func = nn.Linear(in_channels + out_channels, out_channels)
        
        # Batch normalization for stability
        self.bn = nn.BatchNorm1d(out_channels)
    
    def aggregate(self, messages: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
        """
        Aggregate messages from neighbors using specified operation.
        
        The aggregation function must be PERMUTATION INVARIANT:
        - SUM: Commutative operation ✓
        - MEAN: Invariant to order ✓
        - MAX: Commutative operation ✓
        
        Args:
            messages: (E, out_channels) tensor of messages
            indices: (E,) tensor mapping each message to target node
            
        Returns:
            (V, out_channels) aggregated messages for each node
        """
        num_nodes = indices.max().item() + 1
        
        if self.aggregation == 'sum':
            # SUM aggregation: simple summation of messages
            aggregated = torch.zeros(num_nodes, messages.size(1), 
                                    device=messages.device, dtype=messages.dtype)
            aggregated.index_add_(0, indices, messages)
            
        elif self.aggregation == 'mean':
            # MEAN aggregation: average the messages
            # Note: We count degree implicitly via index_add_
            aggregated = torch.zeros(num_nodes, messages.size(1),
                                    device=messages.device, dtype=messages.dtype)
            aggregated.index_add_(0, indices, messages)
            
            # Normalize by degree (number of incoming messages per node)
            degrees = torch.zeros(num_nodes, device=messages.device)
            degrees.index_add_(0, indices, torch.ones(indices.size(0), device=messages.device))
            degrees[degrees == 0] = 1  # Avoid division by zero
            aggregated = aggregated / degrees.unsqueeze(1)
            
        elif self.aggregation == 'max':
            # MAX aggregation: take maximum across messages
            aggregated = torch.zeros(num_nodes, messages.size(1),
                                    device=messages.device, dtype=messages.dtype)
            scatter_max(messages, indices, 0, out=aggregated)
            
        else:
            raise ValueError(f"Unknown aggregation: {self.aggregation}")
        
        return aggregated
    
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of message passing layer.
        
        Mathematical operations:
        
        1. MESSAGE PHASE:
           For each edge (i,j), compute message:
           m_ij = MESSAGE_FUNC(h_j)
        
        2. AGGREGATION PHASE:
           For each node i, aggregate messages from neighbors:
           m_i = AGGREGATE({m_ij : j ∈ N(i)})
        
        3. UPDATE PHASE:
           For each node i, update state:
           h_i' = UPDATE(h_i, m_i)
           h_i' = σ(UPDATE_FUNC([h_i || m_i]))
        
        Args:
            h: (V, in_channels) node features
            edge_index: (2, E) edge indices [source; target]
            
        Returns:
            h_new: (V, out_channels) updated node features
        """
        # Extract source and target node indices
        src, dst = edge_index[0], edge_index[1]
        
        # STEP 1: Prepare messages (transforming neighbor features)
        # For each edge (src, dst), prepare message from src to dst
        messages = self.message_func(h[src])  # (E, out_channels)
        
        # STEP 2: Aggregate messages at target nodes
        # For each node, aggregate messages from all incoming edges
        m = self.aggregate(messages, dst)  # (V, out_channels)
        
        # STEP 3: Update node features combining own features and messages
        # Concatenate original features with aggregated messages
        h_combined = torch.cat([h, m], dim=1)  # (V, in_channels + out_channels)
        h_new = self.update_func(h_combined)   # (V, out_channels)
        h_new = self.bn(h_new)
        h_new = F.relu(h_new)
        
        return h_new

# ============================================================================
# Example: Demonstrating Message Passing
# ============================================================================

print("\n" + "=" * 70)
print("MESSAGE PASSING LAYER IMPLEMENTATION")
print("=" * 70)

# Create a small graph
num_nodes = 5
in_features = 3
out_features = 4

# Random node features: h ∈ ℝ^(V × in_features)
h = torch.randn(num_nodes, in_features)
print(f"\n📊 Input node features shape: {h.shape}")
print(f"   Each node has {in_features} features\n")

# Edge indices: edges from our example
edge_index = torch.tensor([
    [0, 0, 1, 1, 2, 3],  # source nodes
    [1, 2, 2, 3, 3, 4]   # target nodes
], dtype=torch.long)
print(f"📊 Edge index shape: {edge_index.shape}")
print(f"   {edge_index.shape[1]} edges in the graph\n")

# Create message passing layer
mp_layer = MessagePassingLayer(in_features, out_features, aggregation='mean')

# Forward pass
h_out = mp_layer(h, edge_index)

print(f"📊 Output node features shape: {h_out.shape}")
print(f"   Updated feature dimension: {out_features}")
print(f"\n✓ Message passing layer successful!")
print(f"   Each node now incorporates neighbor information!")

# Section 4: Graph Convolutional Networks (GCN)

## 4.1 Mathematical Foundations

### Spectral Graph Convolution

The original motivation for GCN comes from **spectral convolution** on graphs:

**Spectral Convolution Theorem:**

$$y = \sigma(U \Theta_\theta U^T x)$$

Where:
- $U$: Eigenvectors of Laplacian (basis for graph signal processing)
- $\Theta_\theta$: Learnable filter in spectral domain
- $U^T x$: Transform signal to spectral domain
- $\sigma$: Activation function

**Problem**: Computing eigendecomposition is O(V³) - too expensive for large graphs!

### Spatial GCN Formulation (Efficient Alternative)

Instead of using full eigendecomposition, use **Chebyshev polynomial approximation**:

$$H^{(\ell+1)} = \sigma \left( \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(\ell)} W^{(\ell)} \right)$$

Where:
- $\tilde{A} = A + I$ (add self-loops for self-information)
- $\tilde{D}$ = Degree matrix of $\tilde{A}$
- $W^{(\ell)}$ = Learnable weight matrix
- $H^{(\ell)}$ = Node features at layer $\ell$

**Why this works:**
- $D^{-1/2} A D^{-1/2}$ is normalized adjacency (eigenvalues in [0,2])
- Chebyshev approximation with K=1 gives the formula above
- Linear time complexity: O(E)

### Physical Interpretation

$$\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} h$$

This means for each node:
1. Aggregate features from neighbors (via $\tilde{A}$)
2. Include own features (via self-loop in $\tilde{A}$)
3. Normalize by degree (via $\tilde{D}^{-1/2}$) to avoid over-weighting high-degree nodes

In [ ]:
# ============================================================================
# IMPLEMENTATION: Graph Convolutional Network (GCN) Layer
# ============================================================================

class GCNLayer(nn.Module):
    """
    A Graph Convolutional Network layer.
    
    Implements the spatial formulation:
    H' = σ(D^(-1/2) A D^(-1/2) H W)
    
    This layer performs neighborhood aggregation with normalization.
    """
    
    def __init__(self, in_channels: int, out_channels: int, 
                 add_self_loops: bool = True, bias: bool = True):
        """
        Initialize GCN layer.
        
        Args:
            in_channels: Dimension of input features
            out_channels: Dimension of output features
            add_self_loops: Whether to add self-loops to adjacency matrix
            bias: Whether to use bias term
        """
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.add_self_loops = add_self_loops
        
        # Learnable weight matrix W
        self.weight = nn.Parameter(torch.randn(in_channels, out_channels))
        
        # Bias term
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        else:
            self.register_parameter('bias', None)
        
        # Initialize weights using Xavier uniform initialization
        nn.init.xavier_uniform_(self.weight)
        
    def normalize_adjacency(self, edge_index: torch.Tensor, 
                           num_nodes: int) -> torch.Tensor:
        """
        Compute normalized adjacency matrix: D^(-1/2) A D^(-1/2)
        
        Mathematical derivation:
        
        For each edge (i,j) in A:
        - First multiply by D^(-1/2): scale row i by 1/√(d_i)
        - Then multiply by D^(-1/2): scale column j by 1/√(d_j)
        
        This normalization:
        1. Prevents feature explosion from high-degree nodes
        2. Makes eigenvalues bounded in [0, 2]
        3. Improves numerical stability
        
        Args:
            edge_index: (2, E) edge indices
            num_nodes: Number of nodes V
            
        Returns:
            edge_weight: (E,) normalized edge weights
        """
        # Step 1: Add self-loops if requested
        if self.add_self_loops:
            # Create self-loop edges (0,0), (1,1), ..., (n-1, n-1)
            self_loop_edge_index = torch.arange(num_nodes, device=edge_index.device)
            self_loop_edge_index = torch.stack([self_loop_edge_index, self_loop_edge_index])
            
            # Concatenate original edges with self-loops
            edge_index = torch.cat([edge_index, self_loop_edge_index], dim=1)
        
        # Step 2: Compute degree of each node
        # For each node, count how many edges (including self-loops) it has
        degrees = torch.zeros(num_nodes, device=edge_index.device)
        degrees.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1), device=edge_index.device))
        degrees.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1), device=edge_index.device))
        
        # Step 3: Compute D^(-1/2)
        # For each node i: sqrt(1/d_i)
        degrees_inv_sqrt = torch.zeros_like(degrees)
        mask = degrees > 0
        degrees_inv_sqrt[mask] = 1.0 / torch.sqrt(degrees[mask])
        
        # Step 4: Apply normalization to each edge
        # For edge (i,j): weight = D^(-1/2)[i,i] * A[i,j] * D^(-1/2)[j,j]
        #                        = 1/√(d_i) * 1 * 1/√(d_j)
        src, dst = edge_index[0], edge_index[1]
        edge_weight = degrees_inv_sqrt[src] * degrees_inv_sqrt[dst]
        
        return edge_index, edge_weight
    
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of GCN layer.
        
        Implements: H' = σ(D^(-1/2) A D^(-1/2) H W)
        
        Step-by-step breakdown:
        
        1. Transform features: H_transformed = H @ W
           - Project all node features using learnable matrix W
           - Shape: (V, in_channels) @ (in_channels, out_channels) → (V, out_channels)
        
        2. Aggregate from neighbors: H_aggregated = D^(-1/2) A D^(-1/2) H_transformed
           - For each node, aggregate features from neighbors
           - Normalization ensures consistent scale regardless of degree
        
        3. Add bias and apply activation
        
        Args:
            h: (V, in_channels) node features
            edge_index: (2, E) edge indices
            
        Returns:
            h_out: (V, out_channels) updated node features
        """
        num_nodes = h.size(0)
        
        # Step 1: Transform node features using weight matrix
        # h: (V, in_channels) → (V, out_channels)
        h_transformed = h @ self.weight
        
        # Step 2: Compute normalized adjacency matrix
        edge_index_norm, edge_weight = self.normalize_adjacency(edge_index, num_nodes)
        
        # Step 3: Apply graph convolution
        # For each edge, multiply source feature by normalized weight
        src, dst = edge_index_norm[0], edge_index_norm[1]
        
        # Aggregate: for each destination node, sum weighted features from neighbors
        h_aggregated = torch.zeros_like(h_transformed)
        h_aggregated.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.out_channels),
                                 (h_transformed[src] * edge_weight.unsqueeze(1)))
        
        # Step 4: Add bias
        if self.bias is not None:
            h_aggregated = h_aggregated + self.bias
        
        # Step 5: Apply activation function
        h_out = F.relu(h_aggregated)
        
        return h_out

# ============================================================================
# Example: GCN Layer in Action
# ============================================================================

print("\n" + "=" * 70)
print("GRAPH CONVOLUTIONAL NETWORK (GCN) LAYER")
print("=" * 70)

# Create simple graph
num_nodes = 5
in_features = 3
out_features = 4

h = torch.randn(num_nodes, in_features)
edge_index = torch.tensor([
    [0, 0, 1, 1, 2, 3],
    [1, 2, 2, 3, 3, 4]
], dtype=torch.long)

# Initialize GCN layer
gcn = GCNLayer(in_features, out_features, add_self_loops=True)

print(f"\n📊 Input:")
print(f"   • Node features: {h.shape}")
print(f"   • Number of edges: {edge_index.shape[1]}")

# Forward pass
h_out_gcn = gcn(h, edge_index)

print(f"\n📊 Output:")
print(f"   • Node features: {h_out_gcn.shape}")
print(f"   • Features now incorporate neighborhood information!")

# Show mathematical operation
print(f"\n📐 Mathematical Operation:")
print(f"   H' = σ(Ã H W + b)")
print(f"   where:")
print(f"   - Ã = D^(-1/2) (A + I) D^(-1/2)")
print(f"   - W: {in_features} × {out_features} weight matrix")
print(f"   - σ = ReLU activation")

# Section 5: Graph Attention Networks (GAT)

## 5.1 Attention Mechanism for Graphs

**Motivation**: Different neighbors should have different importance!

In GCN, all neighbors contribute equally (after degree normalization). But intuitively:
- A paper by a famous author might be more important
- A well-connected node might be more informative
- Some connections matter more than others

### Mathematical Formulation

For each node $i$ and each neighbor $j \in \mathcal{N}(i)$:

**Step 1: Compute attention coefficients**

$$e_{ij} = \text{LeakyReLU}(a^T [W h_i || W h_j])$$

Where:
- $W h_i$, $W h_j$: Transformed node features
- $a$: Learnable attention vector
- $[·||·]$: Concatenation operator
- LeakyReLU: Non-linear activation

**Step 2: Normalize with softmax**

$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}$$

This ensures attention weights sum to 1 for each node.

**Step 3: Apply attention to aggregate**

$$h_i' = \sigma \left( \sum_{j \in \mathcal{N}(i)} \alpha_{ij} W h_j \right)$$

### Multi-Head Attention

Use multiple attention heads and concatenate results:

$$h_i' = \|_{k=1}^{K} \sigma \left( \sum_{j \in \mathcal{N}(i)} \alpha_{ij}^{(k)} W^{(k)} h_j \right)$$

Where $\|$ denotes concatenation. This increases model capacity.

In [ ]:
# ============================================================================
# IMPLEMENTATION: Graph Attention Network (GAT) Layer
# ============================================================================

class GATLayer(nn.Module):
    """
    Graph Attention Network layer with multi-head attention.
    
    Each head learns a different attention pattern over the neighborhood,
    allowing the model to focus on different types of relationships.
    """
    
    def __init__(self, in_channels: int, out_channels: int, 
                 num_heads: int = 4, dropout: float = 0.0, 
                 concat: bool = True, add_self_loops: bool = True):
        """
        Initialize GAT layer.
        
        Args:
            in_channels: Input feature dimension
            out_channels: Output feature dimension (per head)
            num_heads: Number of attention heads
            dropout: Dropout rate for attention coefficients
            concat: Whether to concatenate heads or average them
            add_self_loops: Whether to add self-loops
        """
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_heads = num_heads
        self.dropout = dropout
        self.concat = concat
        self.add_self_loops = add_self_loops
        
        # For each head, we need:
        # 1. A transformation matrix W
        # 2. An attention vector a
        
        self.weight = nn.Parameter(torch.randn(num_heads, in_channels, out_channels))
        self.att_l = nn.Parameter(torch.randn(num_heads, out_channels, 1))
        self.att_r = nn.Parameter(torch.randn(num_heads, out_channels, 1))
        
        # Bias (only used if concat=True)
        if concat:
            self.bias = nn.Parameter(torch.zeros(num_heads * out_channels))
        else:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        
        # Initialize parameters
        nn.init.xavier_uniform_(self.weight)
        nn.init.xavier_uniform_(self.att_l)
        nn.init.xavier_uniform_(self.att_r)
        
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with multi-head attention.
        
        Process:
        1. Transform node features using weight matrices (one per head)
        2. Compute attention coefficients for each edge
        3. Aggregate using attention weights
        4. Concatenate or average heads
        
        Args:
            h: (V, in_channels) node features
            edge_index: (2, E) edge indices
            
        Returns:
            h_out: (V, num_heads * out_channels) or (V, out_channels)
        """
        num_nodes = h.size(0)
        
        # Add self-loops if requested
        if self.add_self_loops:
            self_loop_index = torch.arange(num_nodes, device=h.device)
            self_loop_index = torch.stack([self_loop_index, self_loop_index])
            edge_index = torch.cat([edge_index, self_loop_index], dim=1)
        
        src, dst = edge_index[0], edge_index[1]
        
        # Initialize output tensor
        if self.concat:
            h_out = torch.zeros(num_nodes, self.num_heads * self.out_channels, 
                               device=h.device, dtype=h.dtype)
        else:
            h_out = torch.zeros(num_nodes, self.out_channels, 
                               device=h.device, dtype=h.dtype)
        
        # Process each attention head
        for head in range(self.num_heads):
            # STEP 1: Transform features for this head
            # h_head: (V, in_channels) → (V, out_channels)
            h_head = h @ self.weight[head]  # (V, out_channels)
            
            # STEP 2: Compute attention logits for each edge
            # For edge (i,j): e_ij = LeakyReLU(a^T [W h_i || W h_j])
            att_src = h_head[src]  # (E, out_channels)
            att_dst = h_head[dst]  # (E, out_channels)
            
            # Attention logits: combine left and right attention vectors
            # e_ij = att_src @ att_l + att_dst @ att_r
            e = att_src @ self.att_l[head] + att_dst @ self.att_r[head]  # (E, 1)
            e = F.leaky_relu(e, negative_slope=0.2)  # (E, 1)
            
            # STEP 3: Normalize with softmax
            # α_ij = softmax_j(e_ij) for each node i
            # This requires grouping edges by destination node
            
            # Initialize attention coefficients
            alpha = torch.zeros(edge_index.size(1), device=h.device, dtype=h.dtype)
            
            # For each node, compute softmax over incoming edges
            for node_idx in range(num_nodes):
                # Find all edges pointing to this node
                mask = dst == node_idx
                e_node = e[mask].squeeze(1)  # Attention logits for this node
                
                if e_node.numel() > 0:
                    # Compute softmax
                    alpha_node = F.softmax(e_node, dim=0)
                    alpha[mask] = alpha_node
            
            # Apply dropout to attention coefficients
            if self.dropout > 0:
                alpha = F.dropout(alpha, p=self.dropout, training=self.training)
            
            # STEP 4: Aggregate using attention weights
            # h'_i = sum_j α_ij W h_j
            h_agg = torch.zeros_like(h_head)
            h_agg.scatter_add_(0, dst.unsqueeze(1).expand(-1, self.out_channels),
                             h_head[src] * alpha.unsqueeze(1))
            
            # STEP 5: Store results
            if self.concat:
                h_out[:, head * self.out_channels:(head+1) * self.out_channels] = h_agg
            else:
                h_out = h_out + h_agg / self.num_heads
        
        # Add bias and apply activation
        h_out = h_out + self.bias
        
        return h_out

# ============================================================================
# Example: GAT Layer
# ============================================================================

print("\n" + "=" * 70)
print("GRAPH ATTENTION NETWORK (GAT) LAYER")
print("=" * 70)

# Create GAT layer with 4 heads
num_nodes = 5
in_features = 3
out_features = 4
num_heads = 4

h = torch.randn(num_nodes, in_features)
edge_index = torch.tensor([
    [0, 0, 1, 1, 2, 3],
    [1, 2, 2, 3, 3, 4]
], dtype=torch.long)

gat = GATLayer(in_features, out_features, num_heads=num_heads, 
              concat=True, dropout=0.1)

h_out_gat = gat(h, edge_index)

print(f"\n📊 Input:")
print(f"   • Node features: {h.shape}")
print(f"   • Number of edges: {edge_index.shape[1]}")

print(f"\n📊 GAT Configuration:")
print(f"   • Input dimension: {in_features}")
print(f"   • Output per head: {out_features}")
print(f"   • Number of heads: {num_heads}")
print(f"   • Total output: {num_heads} × {out_features} = {num_heads * out_features}")

print(f"\n📊 Output:")
print(f"   • Node features: {h_out_gat.shape}")

print(f"\n📐 Key Differences from GCN:")
print(f"   GCN:  All neighbors weighted by 1/√(d_i * d_j)")
print(f"   GAT:  Each neighbor weighted by learned attention α_ij")
print(f"   ✓ More flexible and expressive!")
print(f"   ✓ Can learn importance of different neighbors")
print(f"   ⚠ Higher computational cost due to attention computation")

# Section 6: Complete GNN Model Architecture

## 6.1 Stacking Layers for Multi-Hop Neighborhoods

A key property of GNNs: **Each layer expands the receptive field by one hop**

With $K$ layers, each node receives information from nodes at distance $\leq K$.

### Receptive Field Growth

```
Layer 0: Each node has info from itself (0-hop neighbors)
         Receptive field = {self}
         
Layer 1: Each node aggregates from 1-hop neighbors
         Receptive field = {self} ∪ {1-hop neighbors}
         
Layer 2: Each node gets info from 2-hop neighbors
         (via 1-hop neighbors who got it from 2-hop in layer 1)
         Receptive field = {self} ∪ {1-hop} ∪ {2-hop neighbors}
         
Layer K: Each node has information from all nodes within
         K hops in the graph!
         Receptive field = All nodes within distance K
```

### Expressiveness vs. Over-smoothing Trade-off

**More layers = More expressive?** Not always!

**Over-smoothing problem**:
- With very deep networks (many layers)
- Node features become similar (over-smoothed)
- Network loses ability to distinguish nodes
- Typically 2-4 layers work best in practice

In [ ]:
# ============================================================================
# IMPLEMENTATION: Complete GNN Models
# ============================================================================

class GCNModel(nn.Module):
    """
    Multi-layer Graph Convolutional Network for node classification.
    
    Architecture:
    Input → GCNLayer → ReLU → Dropout →
            GCNLayer → ReLU → Dropout →
            GCNLayer → Output
    """
    
    def __init__(self, in_features: int, hidden_features: int, 
                 num_classes: int, num_layers: int = 2, dropout: float = 0.5):
        """
        Initialize GCN model.
        
        Args:
            in_features: Input feature dimension
            hidden_features: Hidden layer dimension
            num_classes: Number of output classes
            num_layers: Number of GCN layers
            dropout: Dropout rate
        """
        super().__init__()
        self.dropout = dropout
        
        # First layer: input_features → hidden_features
        self.gc1 = GCNLayer(in_features, hidden_features)
        
        # Middle layers: hidden_features → hidden_features
        self.gc_layers = nn.ModuleList([
            GCNLayer(hidden_features, hidden_features) 
            for _ in range(num_layers - 2)
        ])
        
        # Last layer: hidden_features → num_classes
        self.gc_out = GCNLayer(hidden_features, num_classes)
    
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through GCN.
        
        Args:
            h: (V, in_features) node features
            edge_index: (2, E) edge indices
            
        Returns:
            logits: (V, num_classes) class logits
        """
        # First GCN layer + ReLU + Dropout
        h = self.gc1(h, edge_index)
        h = F.dropout(h, p=self.dropout, training=self.training)
        
        # Middle layers
        for gc_layer in self.gc_layers:
            h = gc_layer(h, edge_index)
            h = F.dropout(h, p=self.dropout, training=self.training)
        
        # Output layer (no activation - raw logits)
        logits = self.gc_out(h, edge_index)
        
        return logits


class GATModel(nn.Module):
    """
    Multi-layer Graph Attention Network for node classification.
    
    Uses multi-head attention for more expressive aggregation.
    """
    
    def __init__(self, in_features: int, hidden_features: int,
                 num_classes: int, num_heads: int = 8, num_layers: int = 2,
                 dropout: float = 0.0):
        """
        Initialize GAT model.
        
        Args:
            in_features: Input feature dimension
            hidden_features: Hidden layer dimension
            num_classes: Number of output classes
            num_heads: Number of attention heads
            num_layers: Number of GAT layers
            dropout: Dropout rate
        """
        super().__init__()
        self.dropout = dropout
        
        # First layer with concatenation
        self.gat1 = GATLayer(in_features, hidden_features, 
                            num_heads=num_heads, concat=True, dropout=dropout)
        
        # Middle layers
        self.gat_layers = nn.ModuleList([
            GATLayer(num_heads * hidden_features, hidden_features,
                    num_heads=num_heads, concat=True, dropout=dropout)
            for _ in range(num_layers - 2)
        ])
        
        # Output layer (concatenate then project)
        self.gat_out = GATLayer(num_heads * hidden_features if num_layers > 1 else in_features,
                               num_classes, num_heads=1, concat=False, dropout=dropout)
    
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through GAT.
        
        Args:
            h: (V, in_features) node features
            edge_index: (2, E) edge indices
            
        Returns:
            logits: (V, num_classes) class logits
        """
        # First GAT layer
        h = self.gat1(h, edge_index)
        h = F.dropout(h, p=self.dropout, training=self.training)
        
        # Middle layers
        for gat_layer in self.gat_layers:
            h = gat_layer(h, edge_index)
            h = F.dropout(h, p=self.dropout, training=self.training)
        
        # Output layer
        logits = self.gat_out(h, edge_index)
        
        return logits


# ============================================================================
# Example: Build and Test Models
# ============================================================================

print("\n" + "=" * 70)
print("COMPLETE GNN MODELS")
print("=" * 70)

# Create a larger synthetic graph for testing
num_nodes = 100
in_features = 10
hidden_features = 16
num_classes = 4

# Random features and edges
x = torch.randn(num_nodes, in_features)
edge_index = torch.randint(0, num_nodes, (2, 300))

print(f"\n📊 Synthetic Dataset:")
print(f"   • Nodes: {num_nodes}")
print(f"   • Features per node: {in_features}")
print(f"   • Classes: {num_classes}")
print(f"   • Edges: {edge_index.shape[1]}")

# Build GCN model
print(f"\n🔷 GCN Model:")
gcn_model = GCNModel(in_features, hidden_features, num_classes, 
                    num_layers=3, dropout=0.5)
print(gcn_model)

# Build GAT model
print(f"\n🔶 GAT Model:")
gat_model = GATModel(in_features, hidden_features, num_classes,
                    num_heads=8, num_layers=3, dropout=0.2)
print(gat_model)

# Forward pass
print(f"\n🚀 Forward Pass:")
gcn_output = gcn_model(x, edge_index)
gat_output = gat_model(x, edge_index)

print(f"   • GCN output shape: {gcn_output.shape}")
print(f"   • GAT output shape: {gat_output.shape}")
print(f"   ✓ Models ready for training!")

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Complexity:")
print(f"   • GCN parameters: {count_parameters(gcn_model):,}")
print(f"   • GAT parameters: {count_parameters(gat_model):,}")

# Section 7: Node Classification Pipeline

## 7.1 End-to-End Learning Pipeline

A complete GNN training pipeline consists of:

1. **Data Loading & Preprocessing**: Load graphs, normalize features
2. **Model Initialization**: Set up GNN architecture
3. **Loss Definition**: Choose appropriate loss function
4. **Training Loop**: Optimize model parameters
5. **Validation & Testing**: Evaluate performance
6. **Analysis**: Visualize embeddings and attention patterns

### Loss Functions

For node classification (multi-class):

$$\mathcal{L} = -\frac{1}{|T|} \sum_{i \in T} \sum_{c=1}^{C} y_{ic} \log(\hat{y}_{ic})$$

Where:
- $T$: Training set
- $y_{ic}$: True label (one-hot)
- $\hat{y}_{ic}$: Predicted probability
- $C$: Number of classes

### Training Strategy

**Semi-supervised learning**: Only a small fraction of nodes are labeled

$$L_{total} = L_{labeled} + \lambda L_{unlabeled}$$

Where:
- $L_{labeled}$: Supervised loss on labeled nodes
- $L_{unlabeled}$: Regularization (optional)
- $\lambda$: Trade-off weight

In [ ]:
# ============================================================================
# IMPLEMENTATION: Node Classification Training Pipeline
# ============================================================================

class NodeClassificationTrainer:
    """
    Complete trainer for node classification on graphs.
    
    Handles:
    - Model training with backpropagation
    - Validation and early stopping
    - Metrics computation
    - Training history tracking
    """
    
    def __init__(self, model: nn.Module, device: str = 'cpu'):
        """
        Initialize trainer.
        
        Args:
            model: GNN model to train
            device: 'cpu' or 'cuda'
        """
        self.model = model.to(device)
        self.device = device
        self.history = {
            'train_loss': [], 'train_acc': [],
            'val_loss': [], 'val_acc': [],
            'test_loss': [], 'test_acc': []
        }
    
    def train_epoch(self, x: torch.Tensor, edge_index: torch.Tensor,
                   train_mask: torch.Tensor, y: torch.Tensor,
                   optimizer: torch.optim.Optimizer) -> Tuple[float, float]:
        """
        Train for one epoch.
        
        Args:
            x: Node features
            edge_index: Edge indices
            train_mask: Boolean mask for training nodes
            y: Node labels
            optimizer: PyTorch optimizer
            
        Returns:
            loss, accuracy on training set
        """
        self.model.train()
        
        # Forward pass
        logits = self.model(x, edge_index)
        
        # Compute loss only on training nodes
        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Compute accuracy
        preds = logits[train_mask].argmax(dim=1)
        acc = (preds == y[train_mask]).float().mean().item()
        
        return loss.item(), acc
    
    @torch.no_grad()
    def evaluate(self, x: torch.Tensor, edge_index: torch.Tensor,
                mask: torch.Tensor, y: torch.Tensor) -> Tuple[float, float]:
        """
        Evaluate model on a set of nodes.
        
        Args:
            x: Node features
            edge_index: Edge indices
            mask: Boolean mask for evaluation nodes
            y: Node labels
            
        Returns:
            loss, accuracy on evaluation set
        """
        self.model.eval()
        
        logits = self.model(x, edge_index)
        loss = F.cross_entropy(logits[mask], y[mask]).item()
        preds = logits[mask].argmax(dim=1)
        acc = (preds == y[mask]).float().mean().item()
        
        return loss, acc
    
    def fit(self, x: torch.Tensor, edge_index: torch.Tensor,
           y: torch.Tensor, train_mask: torch.Tensor,
           val_mask: torch.Tensor, test_mask: torch.Tensor,
           epochs: int = 100, lr: float = 0.01, patience: int = 10):
        """
        Full training pipeline with validation and early stopping.
        
        Args:
            x: Node features
            edge_index: Edge indices
            y: Node labels
            train_mask: Training set mask
            val_mask: Validation set mask
            test_mask: Test set mask
            epochs: Maximum training epochs
            lr: Learning rate
            patience: Early stopping patience
        """
        # Move data to device
        x = x.to(self.device)
        edge_index = edge_index.to(self.device)
        y = y.to(self.device)
        train_mask = train_mask.to(self.device)
        val_mask = val_mask.to(self.device)
        test_mask = test_mask.to(self.device)
        
        # Setup optimizer
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        
        best_val_acc = 0
        patience_counter = 0
        
        print(f"\n{'Epoch':<6} {'Train Loss':<12} {'Train Acc':<12} " +
              f"{'Val Loss':<12} {'Val Acc':<12} {'Status':<10}")
        print("-" * 70)
        
        for epoch in range(epochs):
            # Training step
            train_loss, train_acc = self.train_epoch(x, edge_index, train_mask, y, optimizer)
            
            # Validation step
            val_loss, val_acc = self.evaluate(x, edge_index, val_mask, y)
            
            # Store history
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            
            # Early stopping check
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                status = "✓ Best"
            else:
                patience_counter += 1
                status = ""
            
            # Print progress
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"{epoch+1:<6} {train_loss:<12.4f} {train_acc:<12.4f} " +
                      f"{val_loss:<12.4f} {val_acc:<12.4f} {status:<10}")
            
            # Early stopping
            if patience_counter >= patience:
                print(f"\n⛔ Early stopping at epoch {epoch+1}")
                break
        
        # Evaluation on test set
        test_loss, test_acc = self.evaluate(x, edge_index, test_mask, y)
        self.history['test_loss'].append(test_loss)
        self.history['test_acc'].append(test_acc)
        
        print("\n" + "=" * 70)
        print(f"🎯 FINAL RESULTS:")
        print(f"   • Training Accuracy: {train_acc:.4f}")
        print(f"   • Validation Accuracy: {best_val_acc:.4f}")
        print(f"   • Test Accuracy: {test_acc:.4f}")
        print(f"   • Test Loss: {test_loss:.4f}")
        print("=" * 70)
        
        return test_acc

# ============================================================================
# Example: Full Node Classification Pipeline
# ============================================================================

print("\n" + "=" * 70)
print("NODE CLASSIFICATION PIPELINE")
print("=" * 70)

# Create synthetic node classification dataset
torch.manual_seed(42)
num_nodes = 500
num_classes = 4
in_features = 16

# Generate features and labels
x_data = torch.randn(num_nodes, in_features)
y_data = torch.randint(0, num_classes, (num_nodes,))

# Generate sparse graph (power-law degree distribution)
edge_list = []
for node in range(num_nodes):
    # Each node connects to 5-15 random neighbors
    num_neighbors = np.random.randint(5, 15)
    neighbors = np.random.choice(num_nodes, size=num_neighbors, replace=False)
    for neighbor in neighbors:
        if node != neighbor:
            edge_list.append((node, neighbor))

edge_index_data = torch.tensor(edge_list, dtype=torch.long).T
if edge_index_data.numel() == 0:
    edge_index_data = torch.zeros((2, 0), dtype=torch.long)

print(f"\n📊 Synthetic Graph:")
print(f"   • Nodes: {num_nodes}")
print(f"   • Features per node: {in_features}")
print(f"   • Classes: {num_classes}")
print(f"   • Edges: {edge_index_data.shape[1] if edge_index_data.numel() > 0 else 0}")

# Create train/val/test splits (60/20/20)
train_size = int(0.6 * num_nodes)
val_size = int(0.2 * num_nodes)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

indices = np.random.permutation(num_nodes)
train_mask[indices[:train_size]] = True
val_mask[indices[train_size:train_size + val_size]] = True
test_mask[indices[train_size + val_size:]] = True

print(f"\n📊 Train/Val/Test Split:")
print(f"   • Training nodes: {train_mask.sum().item()} ({100*train_mask.sum()/num_nodes:.1f}%)")
print(f"   • Validation nodes: {val_mask.sum().item()} ({100*val_mask.sum()/num_nodes:.1f}%)")
print(f"   • Test nodes: {test_mask.sum().item()} ({100*test_mask.sum()/num_nodes:.1f}%)")

# Initialize and train model
model = GCNModel(in_features, hidden_features=32, num_classes=num_classes,
                num_layers=3, dropout=0.5)
trainer = NodeClassificationTrainer(model, device='cpu')

test_acc = trainer.fit(x_data, edge_index_data, y_data, 
                      train_mask, val_mask, test_mask,
                      epochs=50, lr=0.01, patience=10)

# Section 8: Link Prediction Task

## 8.1 Link Prediction Problem

**Goal**: Predict whether an edge should exist between two nodes.

**Applications**:
- Social network: Friend recommendation
- Knowledge graphs: Relation prediction
- Biological networks: Protein interaction prediction
- Recommender systems: User-item interactions

### Mathematical Formulation

Given a graph $G = (V, E)$, we want to predict missing or future edges.

**Training**: Use known edges as positive examples, sample non-edges as negatives
**Testing**: Score all node pairs and rank by likelihood

### Scoring Functions

For node pair $(i, j)$, compute similarity score:

$$\text{score}(i,j) = h_i^T h_j$$

Or using learned function:

$$\text{score}(i,j) = \text{MLP}([h_i || h_j || h_i \odot h_j])$$

Where:
- $h_i$: Learned embedding of node $i$
- $\odot$: Element-wise multiplication
- $[·||·]$: Concatenation

### Loss Function

**Binary cross-entropy** for each edge:

$$\mathcal{L} = -\frac{1}{|E^+| + |E^-|} \sum_{(i,j) \in E^+ \cup E^-} y_{ij} \log \sigma(s_{ij}) + (1-y_{ij}) \log(1-\sigma(s_{ij}))$$

Where:
- $E^+$: Positive edges (edges to exist)
- $E^-$: Negative edges (non-existent edges)
- $y_{ij} = 1$ if edge exists, 0 otherwise
- $\sigma$: Sigmoid function

### Evaluation Metrics

- **AUC (Area Under ROC Curve)**: Overall ranking quality
- **Precision@K**: Top-K accuracy
- **Mean Reciprocal Rank (MRR)**: Average rank of true edge
- **Hits@K**: Fraction of true edges in top-K

In [ ]:
# ============================================================================
# IMPLEMENTATION: Link Prediction
# ============================================================================

class LinkPredictionModel(nn.Module):
    """
    End-to-end model for link prediction combining GNN encoder with decoder.
    
    Architecture:
    1. GNN Encoder: Learns node embeddings
    2. Link Decoder: Predicts edge probabilities from embeddings
    """
    
    def __init__(self, in_features: int, hidden_features: int, 
                 num_layers: int = 2, architecture: str = 'gcn'):
        """
        Initialize link prediction model.
        
        Args:
            in_features: Input feature dimension
            hidden_features: Hidden dimension
            num_layers: Number of GNN layers
            architecture: 'gcn' or 'gat'
        """
        super().__init__()
        self.architecture = architecture
        
        # GNN Encoder
        if architecture == 'gcn':
            self.encoder = GCNModel(in_features, hidden_features, hidden_features,
                                   num_layers=num_layers, dropout=0.0)
        else:  # gat
            self.encoder = GATModel(in_features, hidden_features, hidden_features,
                                   num_heads=4, num_layers=num_layers, dropout=0.0)
        
        # Link Decoder (MLP)
        self.decoder = nn.Sequential(
            nn.Linear(2 * hidden_features, hidden_features),
            nn.ReLU(),
            nn.Linear(hidden_features, 1)
        )
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
               edge_pairs: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for link prediction.
        
        Args:
            x: Node features
            edge_index: Edge indices (for GNN)
            edge_pairs: (2, M) pairs of nodes to score
            
        Returns:
            scores: (M,) prediction scores for each pair
        """
        # Step 1: Encode nodes to get embeddings
        h = self.encoder(x, edge_index)  # (V, hidden_features)
        
        # Step 2: For each edge pair, get embeddings
        h_i = h[edge_pairs[0]]  # (M, hidden_features)
        h_j = h[edge_pairs[1]]  # (M, hidden_features)
        
        # Step 3: Concatenate embeddings
        h_concat = torch.cat([h_i, h_j], dim=1)  # (M, 2*hidden_features)
        
        # Step 4: Predict edge scores
        scores = self.decoder(h_concat).squeeze(1)  # (M,)
        
        return scores
    
    def get_embeddings(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Get node embeddings.
        
        Args:
            x: Node features
            edge_index: Edge indices
            
        Returns:
            embeddings: (V, hidden_features)
        """
        return self.encoder(x, edge_index)


class LinkPredictionTrainer:
    """
    Trainer for link prediction task.
    """
    
    def __init__(self, model: nn.Module, device: str = 'cpu'):
        self.model = model.to(device)
        self.device = device
        self.history = {'train_loss': [], 'val_auc': [], 'test_auc': []}
    
    @staticmethod
    def compute_auc(scores: torch.Tensor, labels: torch.Tensor) -> float:
        """
        Compute AUC score.
        
        Args:
            scores: Model predictions
            labels: Ground truth (0 or 1)
            
        Returns:
            AUC score
        """
        from sklearn.metrics import roc_auc_score
        
        scores_np = scores.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()
        
        return roc_auc_score(labels_np, scores_np)
    
    def train_epoch(self, x: torch.Tensor, edge_index: torch.Tensor,
                   train_edges_pos: torch.Tensor, train_edges_neg: torch.Tensor,
                   optimizer: torch.optim.Optimizer) -> float:
        """
        Train for one epoch.
        
        Args:
            x: Node features
            edge_index: Edge indices
            train_edges_pos: Positive edges
            train_edges_neg: Negative edges
            optimizer: Optimizer
            
        Returns:
            Loss value
        """
        self.model.train()
        
        # Combine positive and negative edges
        edge_pairs = torch.cat([train_edges_pos, train_edges_neg], dim=1)
        labels = torch.cat([
            torch.ones(train_edges_pos.shape[1]),
            torch.zeros(train_edges_neg.shape[1])
        ]).to(self.device)
        
        # Predictions
        scores = self.model(x, edge_index, edge_pairs)
        
        # Binary cross-entropy loss
        loss = F.binary_cross_entropy_with_logits(scores, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        return loss.item()
    
    @torch.no_grad()
    def evaluate(self, x: torch.Tensor, edge_index: torch.Tensor,
                eval_edges_pos: torch.Tensor, eval_edges_neg: torch.Tensor) -> float:
        """
        Evaluate on validation/test set.
        
        Args:
            x: Node features
            edge_index: Edge indices
            eval_edges_pos: Positive edges
            eval_edges_neg: Negative edges
            
        Returns:
            AUC score
        """
        self.model.eval()
        
        # Combine edges
        edge_pairs = torch.cat([eval_edges_pos, eval_edges_neg], dim=1)
        labels = torch.cat([
            torch.ones(eval_edges_pos.shape[1]),
            torch.zeros(eval_edges_neg.shape[1])
        ])
        
        # Predictions
        scores = self.model(x, edge_index, edge_pairs)
        
        # Compute AUC
        auc = self.compute_auc(scores, labels)
        
        return auc
    
    def fit(self, x: torch.Tensor, edge_index: torch.Tensor,
           train_edges_pos: torch.Tensor, train_edges_neg: torch.Tensor,
           val_edges_pos: torch.Tensor, val_edges_neg: torch.Tensor,
           test_edges_pos: torch.Tensor, test_edges_neg: torch.Tensor,
           epochs: int = 50, lr: float = 0.01, patience: int = 10):
        """
        Full training pipeline.
        """
        # Move to device
        x = x.to(self.device)
        edge_index = edge_index.to(self.device)
        train_edges_pos = train_edges_pos.to(self.device)
        train_edges_neg = train_edges_neg.to(self.device)
        val_edges_pos = val_edges_pos.to(self.device)
        val_edges_neg = val_edges_neg.to(self.device)
        test_edges_pos = test_edges_pos.to(self.device)
        test_edges_neg = test_edges_neg.to(self.device)
        
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        best_val_auc = 0
        patience_counter = 0
        
        print(f"\n{'Epoch':<6} {'Train Loss':<12} {'Val AUC':<12} {'Status':<10}")
        print("-" * 40)
        
        for epoch in range(epochs):
            train_loss = self.train_epoch(x, edge_index, train_edges_pos, 
                                         train_edges_neg, optimizer)
            val_auc = self.evaluate(x, edge_index, val_edges_pos, val_edges_neg)
            
            self.history['train_loss'].append(train_loss)
            self.history['val_auc'].append(val_auc)
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                patience_counter = 0
                status = "✓ Best"
            else:
                patience_counter += 1
                status = ""
            
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"{epoch+1:<6} {train_loss:<12.4f} {val_auc:<12.4f} {status:<10}")
            
            if patience_counter >= patience:
                print(f"\n⛔ Early stopping at epoch {epoch+1}")
                break
        
        # Test evaluation
        test_auc = self.evaluate(x, edge_index, test_edges_pos, test_edges_neg)
        self.history['test_auc'].append(test_auc)
        
        print("\n" + "=" * 70)
        print(f"🎯 LINK PREDICTION RESULTS:")
        print(f"   • Best Validation AUC: {best_val_auc:.4f}")
        print(f"   • Test AUC: {test_auc:.4f}")
        print("=" * 70)
        
        return test_auc


# ============================================================================
# Example: Link Prediction
# ============================================================================

print("\n" + "=" * 70)
print("LINK PREDICTION TASK")
print("=" * 70)

# Create a graph for link prediction
torch.manual_seed(42)
num_nodes_lp = 200
in_features_lp = 10

x_lp = torch.randn(num_nodes_lp, in_features_lp)

# Generate edges
edge_list_full = []
for i in range(num_nodes_lp):
    num_neighbors = np.random.randint(3, 8)
    neighbors = np.random.choice(num_nodes_lp, size=num_neighbors, replace=False)
    for j in neighbors:
        if i != j and (i, j) not in edge_list_full and (j, i) not in edge_list_full:
            edge_list_full.append((i, j))

# Split edges into train/val/test
np.random.shuffle(edge_list_full)
train_val_split = int(0.8 * len(edge_list_full))
val_test_split = int(0.9 * len(edge_list_full))

train_pos = torch.tensor(edge_list_full[:train_val_split]).T
val_pos = torch.tensor(edge_list_full[train_val_split:val_test_split]).T
test_pos = torch.tensor(edge_list_full[val_test_split:]).T

# Use only training edges for GNN
edge_index_lp = train_pos

# Generate negative samples (random non-existent edges)
def generate_negative_samples(pos_edges, num_nodes, num_samples):
    pos_set = set(map(tuple, pos_edges.T.tolist()))
    neg_edges = []
    while len(neg_edges) < num_samples:
        i = np.random.randint(0, num_nodes)
        j = np.random.randint(0, num_nodes)
        if i != j and (i, j) not in pos_set and (j, i) not in pos_set:
            neg_edges.append((i, j))
    return torch.tensor(neg_edges).T

train_neg = generate_negative_samples(train_pos, num_nodes_lp, train_pos.shape[1])
val_neg = generate_negative_samples(val_pos, num_nodes_lp, val_pos.shape[1])
test_neg = generate_negative_samples(test_pos, num_nodes_lp, test_pos.shape[1])

print(f"\n📊 Link Prediction Dataset:")
print(f"   • Nodes: {num_nodes_lp}")
print(f"   • Training edges (pos): {train_pos.shape[1]}")
print(f"   • Validation edges (pos): {val_pos.shape[1]}")
print(f"   • Test edges (pos): {test_pos.shape[1]}")

# Create and train model
lp_model = LinkPredictionModel(in_features_lp, hidden_features=16, 
                              num_layers=2, architecture='gcn')
lp_trainer = LinkPredictionTrainer(lp_model, device='cpu')

test_auc = lp_trainer.fit(x_lp, edge_index_lp, 
                         train_pos, train_neg,
                         val_pos, val_neg,
                         test_pos, test_neg,
                         epochs=30, lr=0.01, patience=10)

# Section 9: Visualization & Interpretation

## 9.1 Embedding Visualization

After training, we want to visualize learned node embeddings in 2D space.

### Dimensionality Reduction Methods

**t-SNE (t-Distributed Stochastic Neighbor Embedding)**:
- Preserves local structure (nearby points stay nearby)
- Good for finding clusters
- Non-linear method
- Slower on large datasets

**UMAP (Uniform Manifold Approximation and Projection)**:
- Preserves both local and global structure
- Faster than t-SNE
- Better for large graphs
- More theoretically grounded

### Mathematical Insight

Both methods solve:
$$\min_{\mathbf{Y}} \text{KL}(P || Q)$$

Where:
- $P$: Similarity in high-dimensional space
- $Q$: Similarity in low-dimensional space
- KL: Kullback-Leibler divergence (difference between distributions)

## 9.2 Attention Visualization

For GAT models, we can visualize which edges the model pays attention to.

**Attention weights**: $\alpha_{ij}$ shows how much node $i$ attends to node $j$

**Interpretation**:
- High $\alpha_{ij}$: Node $i$ considers node $j$ important
- Can reveal learned graph structure
- May differ from original edges
- Multiple heads learn different patterns

In [ ]:
# ============================================================================
# VISUALIZATION: Embeddings and Patterns
# ============================================================================

from sklearn.manifold import TSNE

# Get embeddings from trained model
print("\n" + "=" * 70)
print("EMBEDDING VISUALIZATION")
print("=" * 70)

with torch.no_grad():
    # Get embeddings from node classification model
    embeddings = gcn_model(x_data, edge_index_data).detach().cpu().numpy()

print(f"\n📊 Embedding Statistics:")
print(f"   • Shape: {embeddings.shape}")
print(f"   • Mean: {embeddings.mean():.4f}")
print(f"   • Std: {embeddings.std():.4f}")
print(f"   • Min: {embeddings.min():.4f}")
print(f"   • Max: {embeddings.max():.4f}")

# Reduce to 2D for visualization using t-SNE
print(f"\n🔄 Applying t-SNE dimensionality reduction...")
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Colored by predicted class
ax1 = axes[0]
scatter = ax1.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=y_data.numpy(), cmap='tab10', s=50, alpha=0.6, edgecolors='k')
ax1.set_xlabel('t-SNE Component 1', fontsize=12)
ax1.set_ylabel('t-SNE Component 2', fontsize=12)
ax1.set_title('Node Embeddings (Colored by True Label)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=ax1, label='Class')

# Plot 2: Colored by train/val/test
ax2 = axes[1]
colors = np.zeros(num_nodes)
colors[train_mask] = 1  # Training
colors[val_mask] = 2    # Validation
colors[test_mask] = 3   # Test

color_map = {1: 'blue', 2: 'orange', 3: 'green'}
color_labels = {1: 'Train', 2: 'Val', 3: 'Test'}

for class_idx in [1, 2, 3]:
    mask = colors == class_idx
    ax2.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
               label=color_labels[class_idx], s=50, alpha=0.6, edgecolors='k')

ax2.set_xlabel('t-SNE Component 1', fontsize=12)
ax2.set_ylabel('t-SNE Component 2', fontsize=12)
ax2.set_title('Node Embeddings (Colored by Dataset Split)', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"✓ Embeddings visualized!")
print(f"\n💡 Interpretation:")
print(f"   • Nodes of the same class form clusters (semantic similarity)")
print(f"   • The GNN has learned meaningful representations")
print(f"   • Spatial proximity ≈ semantic similarity")

In [ ]:
# ============================================================================
# VISUALIZATION: Training Dynamics
# ============================================================================

print("\n" + "=" * 70)
print("TRAINING DYNAMICS ANALYSIS")
print("=" * 70)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss curves
ax1 = axes[0]
epochs_range = range(len(trainer.history['train_loss']))
ax1.plot(epochs_range, trainer.history['train_loss'], 'o-', label='Training Loss', linewidth=2)
ax1.plot(epochs_range, trainer.history['val_loss'], 's-', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss vs Training Epochs', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: Accuracy curves
ax2 = axes[1]
ax2.plot(epochs_range, trainer.history['train_acc'], 'o-', label='Training Accuracy', linewidth=2)
ax2.plot(epochs_range, trainer.history['val_acc'], 's-', label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Accuracy vs Training Epochs', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✓ Training dynamics visualized!")
print(f"\n💡 Key Observations:")
print(f"   • Training loss: {trainer.history['train_loss'][0]:.4f} → {trainer.history['train_loss'][-1]:.4f}")
print(f"   • Validation accuracy improves with training")
print(f"   • Gap between train and val indicates regularization need")

# Section 10: Summary & Key Insights

## 10.1 What We've Learned

### Graph Neural Networks Pipeline

```
1. GRAPH DATA
   ├─ Nodes (features, labels)
   └─ Edges (connectivity)
        ↓
        
2. REPRESENTATION
   ├─ Adjacency matrix A
   ├─ Degree matrix D
   ├─ Laplacian L = D - A
   └─ Node features h
        ↓
        
3. MESSAGE PASSING FRAMEWORK
   ├─ Aggregate: m_i = AGG({h_j : j ∈ N(i)})
   └─ Update: h'_i = UPDATE(h_i, m_i)
        ↓
        
4. GNN ARCHITECTURES
   ├─ GCN: Normalized adjacency aggregation
   ├─ GAT: Learned attention aggregation
   └─ GraphSAGE: Neighbor sampling + MLP
        ↓
        
5. TASK-SPECIFIC LAYERS
   ├─ Node Classification: Output layer with softmax
   ├─ Link Prediction: Pairwise scoring function
   └─ Graph Classification: Global pooling + MLP
        ↓
        
6. OPTIMIZATION
   ├─ Forward pass (prediction)
   ├─ Loss computation (task loss)
   ├─ Backward pass (gradient)
   └─ Parameter update (SGD, Adam, etc.)
        ↓
        
7. EVALUATION
   ├─ Accuracy / AUC / Precision / Recall
   └─ Visualizations (embeddings, attention)
```

## 10.2 Mathematical Summary

### Core Equation: Message Passing

$$h_i^{(\ell)} = \text{UPDATE}^{(\ell)} \left( h_i^{(\ell-1)}, \text{AGGREGATE}^{(\ell)} \left( \{ h_j^{(\ell-1)} : j \in \mathcal{N}(i) \} \right) \right)$$

### Specific Implementations

**GCN (Spatial)**:
$$H^{(\ell)} = \sigma \left( D^{-1/2} A D^{-1/2} H^{(\ell-1)} W^{(\ell)} \right)$$

**GAT (Attention)**:
$$h_i^{(\ell)} = \sigma \left( \sum_{j \in \mathcal{N}(i)} \alpha_{ij}^{(\ell)} W^{(\ell)} h_j^{(\ell-1)} \right)$$

Where $\alpha_{ij} = \frac{\exp(a^T[\text{LReLU}(W[h_i||h_j])])}{\sum_k \exp(a^T[\text{LReLU}(W[h_i||h_k])])}$

## 10.3 Key Insights

### ✅ Advantages of GNNs
1. **Inductive learning**: Learn from graph structure, generalize to unseen nodes
2. **Parameter sharing**: Same weights across different nodes
3. **Interpretable**: Attention weights show which edges matter
4. **Scalable**: Linear in number of edges O(E)
5. **Multi-task**: Same model for node/edge/graph tasks

### ⚠️ Challenges
1. **Over-smoothing**: Deep networks lose node distinctiveness
2. **Heterophily**: Poor on graphs where similar nodes aren't connected
3. **Scalability**: Memory-intensive for very large graphs
4. **Over-squashing**: Information bottleneck in deep architectures

## 10.4 Practical Tips

### Model Selection
- **Small graphs (<10K nodes)**: Use GAT (more expressive)
- **Large sparse graphs**: Use GCN (simpler, faster)
- **Heterogeneous graphs**: Use specialized architectures

### Hyperparameter Tuning
- **Depth**: Usually 2-4 layers (avoid over-smoothing)
- **Width**: 64-512 hidden features (task-dependent)
- **Dropout**: 0.0-0.7 (more for larger models)
- **Learning rate**: 0.001-0.1 (use warm-up)

### Common Mistakes
1. ❌ Using too many layers (over-smoothing)
2. ❌ Insufficient dropout (overfitting)
3. ❌ Training on full batch (memory issues)
4. ❌ Not normalizing features
5. ❌ Using same random seed (lucky initialization)

## 10.5 Advanced Topics to Explore

1. **GraphSAGE**: Inductive learning with mini-batch training
2. **Heterogeneous GNNs**: Multi-type nodes and edges
3. **Temporal GNNs**: Dynamic graphs changing over time
4. **Graph Pooling**: Hierarchical graph representations
5. **Equivariant GNNs**: Incorporating symmetries
6. **Explainable GNNs**: Understanding model decisions

In [ ]:
# ============================================================================
# COMPREHENSIVE REFERENCE: Quick Implementation Guide
# ============================================================================

print("\n" + "=" * 70)
print("QUICK REFERENCE: GNN Implementation Checklist")
print("=" * 70)

reference = """
╔══════════════════════════════════════════════════════════════════════════╗
║                   GNN IMPLEMENTATION CHECKLIST                           ║
╚══════════════════════════════════════════════════════════════════════════╝

1. DATA PREPARATION
   ─────────────────
   ✓ Load graph data (nodes, edges, features, labels)
   ✓ Create edge_index: shape (2, E) with node indices
   ✓ Normalize node features: (features - mean) / std
   ✓ Create train/val/test splits (stratified)
   ✓ Handle edge cases (isolated nodes, self-loops, etc.)
   
   Code template:
   ```python
   edge_index = torch.tensor(edges, dtype=torch.long).T  # (2, E)
   features = (features - features.mean()) / features.std()
   ```

2. MODEL ARCHITECTURE
   ──────────────────
   ✓ Choose architecture: GCN, GAT, GraphSAGE, etc.
   ✓ Set layer depths: typically 2-4 layers
   ✓ Configure hidden dimensions: balance expressiveness vs computation
   ✓ Add regularization: dropout, batch norm, L2
   ✓ Initialize weights: Xavier uniform/normal
   
   Code template:
   ```python
   model = GCNModel(
       in_features=x.shape[1],
       hidden_features=64,
       num_classes=num_classes,
       num_layers=3,
       dropout=0.5
   )
   ```

3. TRAINING SETUP
   ───────────────
   ✓ Choose optimizer: Adam (adaptive LR) or SGD (simpler)
   ✓ Set learning rate: 0.001-0.01 for graphs
   ✓ Select loss: CrossEntropyLoss (classification), MSELoss (regression)
   ✓ Implement early stopping: monitor validation metric
   ✓ Track metrics: accuracy, AUC, precision, recall
   
   Code template:
   ```python
   optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
   loss_fn = torch.nn.CrossEntropyLoss()
   ```

4. TRAINING LOOP
   ──────────────
   ✓ Forward pass: logits = model(features, edge_index)
   ✓ Compute loss: loss = loss_fn(logits[train_mask], labels[train_mask])
   ✓ Backward pass: optimizer.zero_grad() → loss.backward() → optimizer.step()
   ✓ Validate: check on validation set
   ✓ Early stopping: stop if val_metric doesn't improve
   
   Code template:
   ```python
   for epoch in range(num_epochs):
       model.train()
       logits = model(x, edge_index)
       loss = loss_fn(logits[train_mask], y[train_mask])
       optimizer.zero_grad()
       loss.backward()
       optimizer.step()
   ```

5. EVALUATION
   ──────────
   ✓ Set model to eval mode: model.eval()
   ✓ No gradient computation: with torch.no_grad():
   ✓ Compute metrics: accuracy, AUC, F1, precision, recall
   ✓ Print results with proper formatting
   ✓ Analyze errors: which nodes are misclassified?
   
   Code template:
   ```python
   model.eval()
   with torch.no_grad():
       logits = model(x, edge_index)
       preds = logits[test_mask].argmax(dim=1)
       acc = (preds == y[test_mask]).float().mean()
   ```

6. VISUALIZATION
   ──────────────
   ✓ t-SNE/UMAP embedding visualization
   ✓ Attention weight heatmaps (for GAT)
   ✓ Training curves (loss, accuracy vs epoch)
   ✓ Confusion matrices
   ✓ Graph structure visualization
   
   Code template:
   ```python
   embeddings = model(x, edge_index).detach().cpu().numpy()
   embeddings_2d = TSNE(n_components=2).fit_transform(embeddings)
   plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=labels)
   ```

7. DEBUGGING TIPS
   ───────────────
   ✓ Check shapes: print(x.shape), print(edge_index.shape)
   ✓ Verify connectivity: ensure edges are valid
   ✓ Monitor gradients: check for NaN or explosions
   ✓ Validate on small dataset first
   ✓ Compare with baselines (MLP, random classifiers)
   ✓ Ablation studies: remove layers/components
   ✓ Check overfitting: gap between train/val losses
   
╔══════════════════════════════════════════════════════════════════════════╗
║                        COMMON ARCHITECTURES                              ║
╚══════════════════════════════════════════════════════════════════════════╝

GCN (Graph Convolutional Network)
─────────────────────────────────
✓ Pro: Simple, efficient, well-understood
✓ Con: Less expressive than attention-based methods
✓ Use: Large sparse graphs, well-connected data
✓ Equation: H' = σ(ÃHW) where à = D^(-1/2)AD^(-1/2)

GAT (Graph Attention Network)  
────────────────────────────
✓ Pro: Interpretable via attention, learns importance
✓ Con: Computational cost, more parameters
✓ Use: Small to medium graphs, heterophilic data
✓ Equation: h'_i = σ(Σ_j α_ij W h_j) with learned α_ij

GraphSAGE
─────────
✓ Pro: Inductive learning, mini-batch training
✓ Con: More complex implementation
✓ Use: Dynamic graphs, inductive setting
✓ Aggregators: mean, LSTM, pooling

╔══════════════════════════════════════════════════════════════════════════╗
║                      PERFORMANCE BENCHMARKS                              ║
╚══════════════════════════════════════════════════════════════════════════╝

Paper Citation Networks (Cora, Citeseer, Pubmed):
- GCN: ~81% accuracy
- GAT: ~83% accuracy
- GraphSAGE: ~82% accuracy

Social Networks (OGB-ARXIV):
- GCN: ~71% accuracy
- GAT: ~72% accuracy
- SIGN (scalable): ~73% accuracy

Knowledge Graphs (OGB-MAG):
- GAT: ~52% accuracy
- R-GCN: ~61% accuracy
- RGAT: ~62% accuracy

╔══════════════════════════════════════════════════════════════════════════╗
║                       RESOURCES & REFERENCES                             ║
╚══════════════════════════════════════════════════════════════════════════╝

📚 Papers:
- Kipf & Welling (2017): Semi-Supervised Classification with GCN
- Velickovic et al. (2018): Graph Attention Networks
- Hamilton et al. (2017): Inductive Representation Learning with GraphSAGE

🔧 Libraries:
- PyTorch Geometric: torch_geometric.nn
- DGL: dgl.nn
- Spektral: spektral.layers

📖 Tutorials:
- PyTorch Geometric docs: https://pytorch-geometric.readthedocs.io/
- Graph Learn: https://graphlearn.io/
- Kaggle competitions: Graph-based competitions

🎯 Next Steps:
1. Implement on real datasets (Cora, OGB benchmarks)
2. Experiment with different architectures
3. Try heterogeneous and temporal extensions
4. Deploy models to production
5. Research latest GNN developments
"""

print(reference)

print("\n" + "=" * 70)
print("✅ COURSE COMPLETE!")
print("=" * 70)
print("""
You have learned:
✓ Graph theory fundamentals
✓ Graph representations and data structures  
✓ Message passing framework
✓ Graph Convolutional Networks (GCN)
✓ Graph Attention Networks (GAT)
✓ Node classification pipeline
✓ Link prediction tasks
✓ Embedding visualization
✓ Model interpretation and debugging

Next: Apply these concepts to your own graph datasets!
""")
print("=" * 70)